In [1]:
import json
from pathlib import Path

In [6]:
DATA_DIR = Path("../../data/processed")

TRAIN_IMAGES = DATA_DIR / "train/images"
TRAIN_ANN = DATA_DIR / "train/annotations"

VAL_IMAGES = DATA_DIR / "validation/images"
VAL_ANN = DATA_DIR / "validation/annotations"

YOLO_TRAIN = DATA_DIR / "train/yolo_labels"
YOLO_VAL = DATA_DIR / "validation/yolo_labels"

YOLO_TRAIN.mkdir(exist_ok=True)
YOLO_VAL.mkdir(exist_ok=True)

In [7]:
def convert_to_yolo(json_file, yolo_file, img_width, img_height):
    
    with open(json_file) as f:
        data = json.load(f)
    
    lines = []
    
    for key, item in data.items():
        if key.startswith("item"):
            bbox = item.get("bounding_box")
            class_id = item.get("category_id")
            
            if bbox is None or class_id is None:
                continue
            
            x1, y1, x2, y2 = bbox
            
            # convert to YOLO format
            x_center = ((x1 + x2) / 2) / img_width
            y_center = ((y1 + y2) / 2) / img_height
            w = (x2 - x1) / img_width
            h = (y2 - y1) / img_height
            
            lines.append(f"{class_id - 1} {x_center} {y_center} {w} {h}")
    
    if lines:
        with open(yolo_file, "w") as f:
            f.write("\n".join(lines))


In [8]:
for json_file in list(TRAIN_ANN.glob("*.json"))[:100]:
    
    img_file = TRAIN_IMAGES / f"{json_file.stem}.jpg"
    
    # get image size
    from PIL import Image
    img = Image.open(img_file)
    w, h = img.size
    
    yolo_file = YOLO_TRAIN / f"{json_file.stem}.txt"
    
    convert_to_yolo(json_file, yolo_file, w, h)

print("Sample conversion done for 100 training images.")


Sample conversion done for 100 training images.


In [9]:
# training folder
for json_file in TRAIN_ANN.glob("*.json"):
    img_file = TRAIN_IMAGES / f"{json_file.stem}.jpg"
    img = Image.open(img_file)
    w, h = img.size
    yolo_file = YOLO_TRAIN / f"{json_file.stem}.txt"
    convert_to_yolo(json_file, yolo_file, w, h)

# validation folder
for json_file in VAL_ANN.glob("*.json"):
    img_file = VAL_IMAGES / f"{json_file.stem}.jpg"
    img = Image.open(img_file)
    w, h = img.size
    yolo_file = YOLO_VAL / f"{json_file.stem}.txt"
    convert_to_yolo(json_file, yolo_file, w, h)

print("full yolo conversion done for train and validation")


full yolo conversion done for train and validation


In [10]:
import random
from pathlib import Path

YOLO_TRAIN = Path("../../data/processed/train/yolo_labels")

# pick 5 random label files
samples = random.sample(list(YOLO_TRAIN.glob("*.txt")), 5)

for f in samples:
    print("File:", f.name)
    print(open(f).read())
    print("-" * 30)

File: 037823.txt
7 0.4970326409495549 0.6634958382877527 0.34124629080118696 0.5017835909631391
1 0.478486646884273 0.33115338882282996 0.43471810089020774 0.3579072532699168
------------------------------
File: 139551.txt
0 0.4967948717948718 0.5321384425216317 0.9935897435897436 0.9283065512978986
------------------------------
File: 016291.txt
11 0.57109375 0.6900826446280992 0.3421875 0.5179063360881543
------------------------------
File: 143922.txt
11 0.49893162393162394 0.4342948717948718 0.9978632478632479 0.8493589743589743
------------------------------
File: 098677.txt
3 0.6153846153846154 0.639225181598063 0.7564102564102564 0.7167070217917676
------------------------------
